# CV Exploration: Catcross + l2_leaf_reg

**CV only — no auto-submit. Check OOF before deciding to use last slot.**

Same as `s6e2-divye-fe-catcross.ipynb` (best LB 0.95382) but with **l2_leaf_reg=3**.

Rationale: the 15 cross-cat pair features create new categorical groups with smaller sizes
than the original features (pair group average ~52k vs individual features up to 450k rows).
CatBoost's ordered TE on smaller groups is noisier — mild l2 regularization makes those
TE estimates more conservative and potentially more robust on the private LB (189k rows).

l2=10 gave +0.00004 CV on the base model (within noise, but positive direction).
l2=3 is gentler — less likely to over-regularize the large original groups while
still stabilizing the smaller cross-cat group estimates.

Baseline: catboost_divye_fe_catcross OOF=0.95566  LB=0.95382

In [1]:
import subprocess
import numpy as np
import pandas as pd
from itertools import combinations
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

DATA_DIR = Path('data')
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
ss    = pd.read_csv(DATA_DIR / 'sample_submission.csv')

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    if 'heart_disease' in df.columns:
        df['heart_disease'] = df['heart_disease'].map({'Absence': 0, 'Presence': 1})
    return df

train = prep(train)
test  = prep(test)

CAT_FEATURES = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
                'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']
NUM_FEATURES = ['age', 'bp', 'cholesterol', 'max_hr', 'st_depression']
ALL_FEATURES = CAT_FEATURES + NUM_FEATURES

X      = train[ALL_FEATURES]
y      = train['heart_disease'].values
X_test = test[ALL_FEATURES]

SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f'Train: {X.shape}  Test: {X_test.shape}')

Train: (630000, 13)  Test: (270000, 13)


In [2]:
# --- Divye FE functions (same as base notebook) ---

def compute_freq_features(train_df, test_df, cols):
    combined = pd.concat([train_df[cols], test_df[cols]])
    tr_freq, te_freq = {}, {}
    for col in cols:
        freq_map = combined[col].value_counts(normalize=True)
        tr_freq[f'{col}_freq'] = train_df[col].map(freq_map).fillna(0).values
        te_freq[f'{col}_freq'] = test_df[col].map(freq_map).fillna(0).values
    return tr_freq, te_freq


def compute_oof_te(train_df, test_df, cols, y, skf):
    global_mean = y.mean()
    oof_te  = {f'{col}_te': np.zeros(len(train_df)) for col in cols}
    test_te = {f'{col}_te': np.zeros(len(test_df))  for col in cols}
    for col in cols:
        key, tr_vals, te_vals, fold_test = f'{col}_te', train_df[col].values, test_df[col].values, []
        for tr_idx, val_idx in skf.split(train_df, y):
            te_map = pd.DataFrame({'v': tr_vals[tr_idx], 't': y[tr_idx]}).groupby('v')['t'].mean()
            oof_te[key][val_idx] = pd.Series(tr_vals[val_idx]).map(te_map).fillna(global_mean).values
            fold_test.append(pd.Series(te_vals).map(te_map).fillna(global_mean).values)
        test_te[key] = np.mean(fold_test, axis=0)
    return oof_te, test_te


def make_interaction_features(te_dict, top_cols):
    return {f'{c1}_x_{c2}': te_dict[c1] * te_dict[c2]
            for c1, c2 in combinations(top_cols, 2)}


def add_cross_cat_features(df, top_cat_cols):
    """Add pairwise string-concatenated categorical columns for CatBoost's internal ordered TE."""
    df = df.copy()
    new_cols = []
    for c1, c2 in combinations(top_cat_cols, 2):
        name = f'{c1}__x__{c2}'
        df[name] = df[c1].astype(str) + '_' + df[c2].astype(str)
        new_cols.append(name)
    return df, new_cols


def build_augmented(base_df, *dicts):
    df = base_df.copy().reset_index(drop=True)
    for d in dicts:
        for name, vals in d.items():
            df[name] = vals
    return df


# Precompute OOF TE to select top-8 for numeric interactions (same as base)
print('Frequency encoding...')
tr_freq, te_freq = compute_freq_features(X, X_test, ALL_FEATURES)

print('OOF target encoding...')
oof_te, test_te = compute_oof_te(X, X_test, ALL_FEATURES, y, SKF)

corrs = {col: abs(np.corrcoef(vals, y)[0, 1]) for col, vals in oof_te.items()}
top8  = sorted(corrs, key=corrs.get, reverse=True)[:8]
print(f'Top-8 for numeric interactions: {top8}')

tr_inter = make_interaction_features(oof_te,  top8)
te_inter = make_interaction_features(test_te, top8)

# Top-6 CATEGORICAL features by TE correlation → cross-cat pairs
top_cat = [col.replace('_te','') for col in top8 if col.replace('_te','') in CAT_FEATURES]
print(f'Top-{len(top_cat)} cat features for cross-cat: {top_cat}')
print(f'Cross-cat pairs: C({len(top_cat)},2) = {len(list(combinations(top_cat,2)))}')

Frequency encoding...
OOF target encoding...
Top-8 for numeric interactions: ['thallium_te', 'chest_pain_type_te', 'max_hr_te', 'number_of_vessels_fluro_te', 'st_depression_te', 'exercise_angina_te', 'slope_of_st_te', 'sex_te']
Top-6 cat features for cross-cat: ['thallium', 'chest_pain_type', 'number_of_vessels_fluro', 'exercise_angina', 'slope_of_st', 'sex']
Cross-cat pairs: C(6,2) = 15


In [3]:
CATBOOST_PARAMS = dict(
    iterations        = 2000,
    learning_rate     = 0.05,
    depth             = 6,
    l2_leaf_reg       = 3,       # <-- mild regularization for cross-cat group stability
    task_type         = 'CPU',
    thread_count      = -1,
    random_seed       = 42,
    verbose           = 0,
)

global_mean = y.mean()
oof_preds   = np.zeros(len(y))
test_folds  = np.zeros((5, len(X_test)))

for fold, (tr_idx, val_idx) in enumerate(SKF.split(X, y)):
    print(f'\n=== Fold {fold + 1}/5 ===')
    X_tr_base  = X.iloc[tr_idx].reset_index(drop=True)
    X_val_base = X.iloc[val_idx].reset_index(drop=True)
    y_tr, y_val = y[tr_idx], y[val_idx]

    # Freq encoding
    fold_tr_freq, fold_te_freq  = compute_freq_features(X_tr_base, X_test,    ALL_FEATURES)
    _,            fold_val_freq = compute_freq_features(X_tr_base, X_val_base, ALL_FEATURES)

    # Per-fold TE
    fold_tr_te, fold_val_te, fold_te_te = {}, {}, {}
    for col in ALL_FEATURES:
        key    = f'{col}_te'
        te_map = pd.DataFrame({'v': X_tr_base[col].values, 't': y_tr}).groupby('v')['t'].mean()
        fold_tr_te[key]  = X_tr_base[col].map(te_map).fillna(global_mean).values
        fold_val_te[key] = X_val_base[col].map(te_map).fillna(global_mean).values
        fold_te_te[key]  = X_test[col].map(te_map).fillna(global_mean).values

    # Numeric interactions
    fold_tr_inter  = make_interaction_features(fold_tr_te,  top8)
    fold_val_inter = make_interaction_features(fold_val_te, top8)
    fold_te_inter  = make_interaction_features(fold_te_te,  top8)

    # Build augmented frames
    X_tr_aug  = build_augmented(X_tr_base,  fold_tr_freq,  fold_tr_te,  fold_tr_inter)
    X_val_aug = build_augmented(X_val_base, fold_val_freq, fold_val_te, fold_val_inter)
    X_te_aug  = build_augmented(X_test,     fold_te_freq,  fold_te_te,  fold_te_inter)

    # Add cross-cat features
    X_tr_aug,  cross_cols = add_cross_cat_features(X_tr_aug,  top_cat)
    X_val_aug, _          = add_cross_cat_features(X_val_aug, top_cat)
    X_te_aug,  _          = add_cross_cat_features(X_te_aug,  top_cat)

    all_cat = CAT_FEATURES + cross_cols

    m = CatBoostClassifier(**CATBOOST_PARAMS, cat_features=all_cat)
    m.fit(X_tr_aug, y_tr, eval_set=(X_val_aug, y_val), early_stopping_rounds=50)

    oof_preds[val_idx] = m.predict_proba(X_val_aug)[:, 1]
    test_folds[fold]   = m.predict_proba(X_te_aug)[:, 1]
    print(f'Fold {fold+1}  AUC={roc_auc_score(y_val, oof_preds[val_idx]):.5f}  best_iter={m.best_iteration_}')

cv_auc     = roc_auc_score(y, oof_preds)
test_preds = test_folds.mean(axis=0)

print(f'\nOOF AUC (catcross_l2=3): {cv_auc:.5f}')
print(f'catcross baseline:        0.95566')
print(f'Delta:                    {cv_auc - 0.95566:+.5f}')


=== Fold 1/5 ===
Fold 1  AUC=0.95607  best_iter=604

=== Fold 2/5 ===
Fold 2  AUC=0.95492  best_iter=503

=== Fold 3/5 ===
Fold 3  AUC=0.95575  best_iter=574

=== Fold 4/5 ===
Fold 4  AUC=0.95539  best_iter=650

=== Fold 5/5 ===
Fold 5  AUC=0.95620  best_iter=600

OOF AUC (catcross_l2=3): 0.95566
catcross baseline:        0.95566
Delta:                    +0.00000


In [4]:
# --- SAVE CSV (no submit) ---
# Run All stops here. Check OOF delta, then manually run cell-05 to submit.

import subprocess

label = 'catboost_divye_fe_catcross_l2'
fname = f'submissions/{label}.csv'
sub = ss.copy()
sub['Heart Disease'] = test_preds
sub.to_csv(fname, index=False)
print(f'Saved: {fname}')
print(f'OOF: {cv_auc:.5f}  (delta vs catcross: {cv_auc - 0.95566:+.5f})')
print()
print('To submit, run cell-05.')

Saved: submissions/catboost_divye_fe_catcross_l2.csv
OOF: 0.95566  (delta vs catcross: +0.00000)

To submit, run cell-05.


In [5]:
# --- SUBMIT — run manually only if you decide to use the slot ---

desc = f'{label} | cv_auc={cv_auc:.4f}'
r = subprocess.run(
    ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
     '-f', fname, '-m', desc],
    capture_output=True, text=True
)
status = 'submitted' if 'successfully' in r.stdout.lower() else r.stdout.strip()[:120]
print(f'{label}: {status}')

catboost_divye_fe_catcross_l2: 400 Client Error: Bad Request for url: https://www.kaggle.com/api/v1/competitions/submissions/submit/playground-series-s
